# 🎬 Cinematic Engine V17

### شغّل الـ cells بالترتيب — 1 ثم 2 ثم 3

**قبل أي حاجة:** `Runtime → Change runtime type → T4 GPU → Save`


In [ ]:
# ════════════════════════════════════════
# CELL 1 — SETUP (شغّله مرة واحدة بس)
# ════════════════════════════════════════
import os, subprocess, sys

# تأكد من GPU
import torch
if not torch.cuda.is_available():
    raise SystemExit('❌ مفيش GPU — روح Runtime → Change runtime type → T4 GPU')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# تحميل المشروع
REPO = 'https://github.com/michelluke1984-cyber/cinematic-engine'
DIR  = '/content/cinematic-engine'

if not os.path.exists(DIR):
    print('⬇️ جاري تحميل المشروع...')
    os.system(f'git clone -q {REPO} {DIR}')
    print('✅ تم تحميل المشروع')
else:
    print('✅ المشروع موجود — جاري التحديث...')
    os.system(f'cd {DIR} && git pull -q')

os.chdir(DIR)

# تثبيت الـ packages
print('\n📦 جاري تثبيت الـ packages...')
cmds = [
    'pip install -q diffusers transformers accelerate xformers einops',
    'pip install -q gfpgan realesrgan nltk sentencepiece tqdm',
    'pip install -q "gradio>=4.0.0" bitsandbytes Pillow',
    'pip install -q insightface onnxruntime-gpu',
    'pip install -q websockets nest_asyncio pyngrok',
]
for cmd in cmds:
    subprocess.run(cmd, shell=True, capture_output=True)
    print('.', end='', flush=True)
print(' ✅')

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# تحميل الـ model weights
print('\n⬇️ جاري تحميل الـ models...')
models = [
    ('GFPGANv1.3.pth',           'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth'),
    ('realesr-general-x4v3.pth', 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth'),
]
for fname, url in models:
    if not os.path.exists(fname):
        subprocess.run(f'wget -q -O {fname} {url}', shell=True)
        print(f'✅ {fname}')
    else:
        print(f'✅ {fname} موجود')

print('\n' + '='*40)
print('✅ CELL 1 خلص — شغّل CELL 2')
print('='*40)

In [ ]:
# ════════════════════════════════════════
# CELL 2 — تحميل الـ Engine
# ════════════════════════════════════════
import importlib.util, sys, os, torch

DIR = '/content/cinematic-engine'
os.chdir(DIR)

ENGINE = os.path.join(DIR, 'cinematic_engine_v16_pro.py')
if not os.path.exists(ENGINE):
    raise SystemExit(f'❌ الملف مش موجود: {ENGINE}\nشغّل CELL 1 الأول')

print('⏳ جاري تحميل CinematicEngineV16...')

# تحميل وتنفيذ الـ module
spec  = importlib.util.spec_from_file_location('cev16', ENGINE)
cev16 = importlib.util.module_from_spec(spec)
sys.modules['cev16'] = cev16
spec.loader.exec_module(cev16)  # لازم يتنفذ الأول

# نقل كل الـ classes للـ global scope
for k, v in vars(cev16).items():
    if not k.startswith('_'):
        globals()[k] = v
        sys.modules['__main__'].__dict__[k] = v

if 'CinematicEngineV16' not in globals():
    raise SystemExit('❌ CinematicEngineV16 مش موجودة في الملف')

# إنشاء الـ engine
engine = CinematicEngineV16()
globals()['engine'] = engine
sys.modules['__main__'].__dict__['engine'] = engine

print(f'✅ Engine جاهز')
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print('\n' + '='*40)
print('✅ CELL 2 خلص — شغّل CELL 3')
print('='*40)

In [ ]:
# ════════════════════════════════════════
# CELL 3 — تشغيل الـ Bridge
# (هيفضل شغّال — اضغط ⬛ عشان توقف)
# ════════════════════════════════════════
import importlib.util, sys, os, nest_asyncio
nest_asyncio.apply()

DIR = '/content/cinematic-engine'
os.chdir(DIR)

# تحرير البورت
os.system('fuser -k 7860/tcp 2>/dev/null || true')
import time; time.sleep(1)

# التأكد من الـ engine
_engine = globals().get('engine') or sys.modules.get('__main__', {}).__dict__.get('engine')
if _engine is None:
    raise SystemExit('❌ شغّل CELL 2 الأول')
print(f'✅ Engine: {type(_engine).__name__}')

# تحميل الـ bridge
BRIDGE = os.path.join(DIR, 'cev17_backend.py')
spec   = importlib.util.spec_from_file_location('bridge', BRIDGE)
bridge = importlib.util.module_from_spec(spec)
sys.modules['bridge'] = bridge
spec.loader.exec_module(bridge)
print('✅ Bridge loaded')

# ngrok tunnel
from pyngrok import conf, ngrok as _ngrok
conf.get_default().auth_token = 'L3NYYCRD5VTGCOML4S2EBWAMDMN6KI2K'
tunnel = _ngrok.connect(7860, 'tcp')
ws_url = tunnel.public_url.replace('tcp://', 'ws://')

print('\n' + '='*50)
print('🌐 الـ URL بتاعك:')
print(f'   {ws_url}')
print('='*50)
print('افتح الـ Dashboard واضغط WS: OFF')
print('حط الـ URL ده واضغط Connect')
print('='*50)
print('⬛ اضغط عشان توقف')
print()

bridge.run_bridge(engine=_engine, use_ngrok=False)